# Summarization

`SummarizationMiddleware` compresses older messages and keeps a recent tail. Compare the original history, the request after summarization, and the final answer. The summary is model-generated and can lose facts. [Documentation](https://docs.langchain.com/oss/python/langchain/middleware/built-in#summarization).

### Trigger conditions and retained context

A tuple such as `("messages", 12)` is one threshold. A dictionary such as `{"messages": 12, "tokens": 200}` requires both thresholds (AND). A list of conditions accepts either condition (OR). Token thresholds use the token counter; fraction thresholds use the model's declared context-window size. A fraction therefore requires a model profile with `max_input_tokens`.

`keep` preserves exactly one kind of recent context: messages, tokens or a fraction. The custom summary prompt determines which facts survive; `trim_tokens_to_summarize` bounds the input to the summarizer. Tool-call/result pairs remain together so the retained conversation stays valid.

In [ ]:
# Imports and project path setup only; examples start in the next cell.
from pathlib import Path
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
LESSONS = ROOT / "notebooks" / "langchain" / "06_middlewares"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(LESSONS) not in sys.path:
    sys.path.insert(0, str(LESSONS))

from lab.data import conversation, prompt
from lab.display import TraceMiddleware, expect_error, show_messages, show_source
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from model_config import get_model, run_config

In [ ]:
# Execute
model = get_model()
summarizer = get_model("summarizer")
messages = conversation("project")
print("BEFORE:", len(messages), "messages")

In [ ]:
# Inspect output (rerun without executing the agent)
show_messages(messages[-8:], pretty=True)

In [ ]:
# Execute
trace = TraceMiddleware("01_summary", pretty=True)
agent = create_agent(
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=summarizer,
            trigger=[{"messages": 12, "tokens": 200}, ("messages", 20)],
            keep=("messages", 4),
            summary_prompt="Preserve names, decisions, owners and deadlines. Summarize only these messages: {messages}",
            trim_tokens_to_summarize=4000,
        ),
        trace,
    ],
)
result = agent.invoke(
    {"messages": messages + [("user", "Ok nice, so tell me what will Julia do and what will Mark do")]},
    config=run_config(),
)

In [ ]:
# Inspect output (rerun without executing the agent)
print("AFTER:", len(result["messages"]), "messages")
show_messages(result["messages"], pretty=True)
assert len(result["messages"]) < len(messages)